# Notebook 03 — LSTM Training
**Week 2 Task:** Train the LSTM price direction model on all 49 Nifty 50 stocks.

Target: validation accuracy > 55% on 2022–2026 holdout data.

Split: Train 2000–2020 | Val 2021 (inside LSTM) | Test 2022–2026

In [ ]:
import os
os.chdir(r'C:\Users\Aryan\Desktop\everything\PROJECTS\200%BOT\ai-trading-bot\ai-trading-bot')
print('Working dir:', os.getcwd())

In [ ]:
import sys
sys.path.append('.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.preprocess import preprocess_symbol, get_feature_columns, get_train_test_split
from src.models.lstm import LSTMModel

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Imports OK')

## Step 1 — Train on RELIANCE first (single stock test)

In [ ]:
df = preprocess_symbol('RELIANCE')
feat_cols = get_feature_columns(df)
train_df, test_df = get_train_test_split(df, test_start='2022-01-01')  # CHANGED: was 2019-01-01

print(f'Features: {len(feat_cols)}')
print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')
print(f'Train period: {train_df.index[0].date()} to {train_df.index[-1].date()}')
print(f'Test period:  {test_df.index[0].date()} to {test_df.index[-1].date()}')

In [ ]:
# Train — will use your RTX 5050 automatically
model = LSTMModel(lookback=60)
history = model.train(train_df, test_df, feat_cols, epochs=50, patience=8)
print('Training complete')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history['train_loss'], label='Train Loss', color='steelblue')
ax1.plot(history['val_loss'], label='Val Loss', color='red')
ax1.set_title('Loss Curves — RELIANCE LSTM')
ax1.legend()
ax1.set_xlabel('Epoch')

ax2.plot(history['val_acc'], label='Val Accuracy', color='green')
ax2.axhline(0.55, color='orange', linestyle='--', label='Target (55%)')
ax2.axhline(0.50, color='red', linestyle='--', label='Random (50%)')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.set_xlabel('Epoch')
ax2.set_ylim(0.4, 0.75)

plt.tight_layout()
os.makedirs('models/results', exist_ok=True)
plt.savefig('models/results/lstm_training_RELIANCE.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Full evaluation on test set
results = model.evaluate(test_df, feat_cols)
print('\nTest Set Results:')
for k, v in results.items():
    print(f'  {k}: {v}')

if results['accuracy'] >= 0.55:
    print('\n✅ Target accuracy achieved — model is viable')
else:
    print('\n⚠️  Below 55% — check features or increase training data')

In [ ]:
# Save the trained model
model.save('lstm_reliance')
print('Model saved to models/saved/')

## Step 2 — Train on ALL stocks and pick the best

In [ ]:
# Train on all stocks, track accuracy per symbol
# This uses your GPU so each stock takes ~2 min

results_all = []
failed = []

raw_files = [f.replace('.csv','') for f in os.listdir('data/raw')
             if f.endswith('.csv') and not f.endswith('.NS.csv')]
skip = {'INFRATEL', 'NIFTY50_all', 'stock_metadata'}
symbols = [s for s in raw_files if s not in skip]

print(f'Training on {len(symbols)} symbols...')
print('This will take ~15-20 minutes on GPU\n')

for i, symbol in enumerate(symbols, 1):
    try:
        df = preprocess_symbol(symbol)
        if df is None or len(df) < 500:
            print(f'[{i}/{len(symbols)}] {symbol}: skipped (insufficient data)')
            continue

        feat_cols = get_feature_columns(df)
        train_df, test_df = get_train_test_split(df, test_start='2022-01-01')  # CHANGED: was 2019-01-01

        if len(test_df) < 100:
            print(f'[{i}/{len(symbols)}] {symbol}: skipped (test set too small)')
            continue

        m = LSTMModel(lookback=60)
        m.train(train_df, test_df, feat_cols, epochs=30, patience=5)
        r = m.evaluate(test_df, feat_cols)
        r['symbol'] = symbol
        results_all.append(r)
        print(f'[{i}/{len(symbols)}] {symbol}: acc={r["accuracy"]:.3f}')

    except Exception as e:
        print(f'[{i}/{len(symbols)}] {symbol}: ERROR — {e}')
        failed.append(symbol)

print(f'\nDone: {len(results_all)} trained | {len(failed)} failed')

In [ ]:
# Results summary
results_df = pd.DataFrame(results_all).sort_values('accuracy', ascending=False)
print('Top 10 stocks by LSTM accuracy:')
print(results_df[['symbol','accuracy','precision_up','recall_up','f1_up']].head(10).to_string(index=False))
print()
print(f'Mean accuracy across all stocks: {results_df["accuracy"].mean():.3f}')
print(f'Stocks above 55% accuracy: {(results_df["accuracy"] >= 0.55).sum()}')

results_df.to_csv('models/results/lstm_all_stocks.csv', index=False)
print('\nSaved to models/results/lstm_all_stocks.csv')

## Step 3 — Train final model on top 10 stocks combined

In [ ]:
# Train final model on top 10 stocks combined (more data = better generalisation)
top10 = results_df.head(10)['symbol'].tolist()
print(f'Training final model on top 10 stocks: {top10}')

combined_train = []
combined_test = []

for symbol in top10:
    df = preprocess_symbol(symbol)
    if df is not None:
        feat_cols = get_feature_columns(df)
        tr, te = get_train_test_split(df, test_start='2022-01-01')  # CHANGED: was 2019-01-01
        combined_train.append(tr)
        combined_test.append(te)

train_combined = pd.concat(combined_train).sort_index()
test_combined = pd.concat(combined_test).sort_index()

print(f'Combined train: {len(train_combined)} rows')
print(f'Combined test:  {len(test_combined)} rows')

final_model = LSTMModel(lookback=60)
history_final = final_model.train(train_combined, test_combined, feat_cols, epochs=50, patience=8)
final_results = final_model.evaluate(test_combined, feat_cols)
print('\nFinal model results:')
print(final_results)

final_model.save('lstm')
print('\n✅ Final LSTM saved to models/saved/lstm.pt')

## ✅ Done

LSTM is trained and saved. Next: open `04_xgboost_training.ipynb`.